In [ ]:
%run log_spiral.ipynb

In [ ]:
import numpy as np


def build_shell_mesh(
    theta,
    r,
    x,
    y,
    aperture_width,
    aperture_height,
    z_scale=0.08,
    aperture_points=32,
    ribbing_enabled=True,
    ribbing_amplitude=0.08,
    ribbing_frequency=10,
    lip_enabled=True,
    lip_start=0.88,
    lip_strength=0.35,
    lip_power=3
):
    """
    Build a 3D shell-like mesh by sweeping an elliptical aperture along a
    logarithmic spiral.

    Conceptually, this treats the shell as a record of growth:

    - The spiral path represents the route followed by the growing aperture.
    - At each point on the spiral, an elliptical aperture is generated.
    - The aperture grows larger as the shell expands.
    - Consecutive apertures are stitched together into a continuous surface.
    - Periodic changes in aperture size create growth ribs.
    - Late-stage enlargement of the aperture creates a flared shell lip.
    - A colour value is assigned to each vertex to simulate shell banding.

    Ribbing controls:

    - Smaller ribbing_amplitude gives subtler ribs.
    - Larger ribbing_amplitude gives stronger raised growth ribs.
    - Smaller ribbing_frequency gives broader, more widely spaced ribs.
    - Larger ribbing_frequency gives finer, more closely spaced ribs.

    Lip controls:

    - lip_start controls when the final aperture flare begins.
      For example, 0.88 means the lip starts forming after 88% of growth.
    - lip_strength controls how much the final aperture expands.
    - lip_power controls how gradually or abruptly the flare develops.

    :param theta: Spiral angle values, representing growth progression
    :param r: Spiral radius values
    :param x: Spiral x coordinates
    :param y: Spiral y coordinates
    :param aperture_width: Width of elliptical aperture at each growth point
    :param aperture_height: Height of elliptical aperture at each growth point
    :param z_scale: Controls vertical rise of the shell through growth
    :param aperture_points: Number of vertices used to draw each aperture ring
    :param ribbing_enabled: Whether to add periodic growth ribs
    :param ribbing_amplitude: Strength of the growth rib modulation
    :param ribbing_frequency: Number/frequency of ribs along the shell
    :param lip_enabled: Whether to flare the final aperture lip
    :param lip_start: Fraction of growth at which lip flare begins
    :param lip_strength: Maximum proportional enlargement of final aperture
    :param lip_power: Shape of the lip flare curve
    :return: X, Y, Z vertex arrays, C colour values, and triangle index arrays I, J, K
    """

    # Number of aperture positions along the growth path.
    # Each one represents a moment in the shell's growth history.
    n_spiral = len(theta)

    # Number of points around each aperture ring.
    # More points give a smoother aperture cross-section.
    n_aperture = aperture_points

    # Angles around the aperture cross-section.
    # These define the elliptical rim of the shell opening at each growth step.
    phi = np.linspace(0, 2*np.pi, n_aperture, endpoint=False)

    # Mesh vertex positions.
    # These will hold every point on every aperture ring.
    X = []
    Y = []
    Z = []

    # Per-vertex colour/intensity values used for shell pigmentation bands.
    C = []

    # ---------------------------------------------------------------------
    # Build the shell as a sequence of aperture rings
    # ---------------------------------------------------------------------
    for idx in range(n_spiral):

        # Centre of the aperture at this growth stage.
        # The aperture follows the logarithmic spiral in x/y.
        cx = x[idx]
        cy = y[idx]

        # Lift the aperture slightly in z as growth progresses.
        # Setting z_scale to 0 gives a flat ammonite/nautilus-like coil.
        cz = z_scale * theta[idx]

        # Convert aperture width/height into semi-axes.
        # These define the basic elliptical shape of the shell opening.
        aw = aperture_width[idx] / 2
        ah = aperture_height[idx] / 2

        # -------------------------------------------------------------
        # Growth ribs
        # -------------------------------------------------------------
        # Real shells often show periodic ridges caused by rhythmic or
        # episodic shell deposition. We simulate that by slightly expanding
        # and contracting the aperture as it moves along the growth path.
        if ribbing_enabled:
            ribbing_factor = 1 + ribbing_amplitude * np.sin(
                ribbing_frequency * theta[idx]
            )
        else:
            ribbing_factor = 1.0

        # Apply the ribbing to the aperture itself.
        # This makes ribs part of the geometry, not just the colouring.
        aw *= ribbing_factor
        ah *= ribbing_factor

        # -------------------------------------------------------------
        # Aperture lip flare
        # -------------------------------------------------------------
        # Mature shells often have a thickened or flared final opening.
        # We simulate that by enlarging the aperture during the final
        # fraction of growth.
        if lip_enabled:
            growth_fraction = idx / (n_spiral - 1)

            if growth_fraction >= lip_start:
                # Normalised progress through the lip-forming phase:
                # 0 at lip_start, 1 at the final aperture.
                lip_t = (growth_fraction - lip_start) / (1 - lip_start)

                # Smoothly increase the aperture size towards the end.
                lip_factor = 1 + lip_strength * lip_t**lip_power
            else:
                lip_factor = 1.0
        else:
            lip_factor = 1.0

        # Apply the final aperture flare.
        aw *= lip_factor
        ah *= lip_factor

        # Rotate the aperture so it follows the coiling direction.
        # This makes each shell opening turn with the spiral rather than
        # remaining fixed in world coordinates.
        angle = theta[idx]

        cos_a = np.cos(angle)
        sin_a = np.sin(angle)

        # -------------------------------------------------------------
        # Generate one elliptical aperture ring
        # -------------------------------------------------------------
        for p in phi:

            # Local aperture coordinates before rotation.
            # local_x runs across the aperture width.
            # local_z runs vertically through the aperture height.
            local_x = aw * np.cos(p)
            local_z = ah * np.sin(p)

            # Rotate the aperture in the x/y plane and place it at the
            # current point on the shell growth path.
            px = cx + local_x * cos_a
            py = cy + local_x * sin_a
            pz = cz + local_z

            X.append(px)
            Y.append(py)
            Z.append(pz)

            # ---------------------------------------------------------
            # Pigmentation banding
            # ---------------------------------------------------------
            # Colour depends partly on growth position theta and partly
            # on position around the aperture p.
            #
            # theta-based variation creates bands laid down during growth.
            # p-based variation creates variation around the aperture,
            # giving streaks/flames rather than perfectly uniform rings.
            #
            # Multiplying the aperture-position term by ribbing_factor
            # makes the pigment slightly stronger on rib crests.
            band = np.sin(8 * theta[idx]) + 0.4 * np.sin(3 * p) * ribbing_factor

            C.append(band)

    # Convert vertex lists into arrays for Plotly.
    X = np.array(X)
    Y = np.array(Y)
    Z = np.array(Z)
    C = np.array(C)

    # Triangle index arrays.
    # Plotly Mesh3d represents a surface as many triangular faces.
    I = []
    J = []
    K = []

    # ---------------------------------------------------------------------
    # Stitch neighbouring aperture rings together
    # ---------------------------------------------------------------------
    # Each pair of consecutive aperture rings forms a band of shell surface.
    # Each quadrilateral patch between two rings is split into two triangles.
    for s in range(n_spiral - 1):
        for p in range(n_aperture):

            # Current aperture ring point
            current = s * n_aperture + p

            # Next point around the same aperture ring
            next_p = s * n_aperture + ((p + 1) % n_aperture)

            # Corresponding point on the next growth-stage aperture ring
            current_next = (s + 1) * n_aperture + p

            # Next point around the next aperture ring
            next_next = (s + 1) * n_aperture + ((p + 1) % n_aperture)

            # First triangle of the shell-surface patch
            I.append(current)
            J.append(current_next)
            K.append(next_p)

            # Second triangle of the shell-surface patch
            I.append(next_p)
            J.append(current_next)
            K.append(next_next)

    return X, Y, Z, C, I, J, K

In [ ]:
def build_chamber_septa(
    theta,
    shell_mesh,
    aperture_points=32,
    chamber_spacing=0.65,
    septum_rings=8,
    septum_depth=0.35
):
    """
    Build curved internal chamber walls inside the shell.

    Conceptually, these are septa: internal partitions laid down at intervals
    as the shell grows. In a Nautilus-like shell, the animal moves forward into
    the newest body chamber, leaving older chambers behind, separated by curved
    walls.

    This function takes the existing shell surface mesh and inserts chamber
    walls at selected aperture rings.

    Each septum is built as a shallow concave bowl:

    - the outer edge matches an existing aperture ring
    - nested smaller rings fill the aperture
    - the centre is pushed slightly backward along the growth direction
    - the result is a curved chamber wall rather than a flat disc

    Chamber colouring is varied by growth age:

    - older inner chambers are darker
    - newer outer chambers are lighter

    :param theta: Spiral angle values, representing growth progression
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_points: Number of points around each aperture ring
    :param chamber_spacing: Angular distance between successive chamber walls
    :param septum_rings: Number of nested rings used to fill each septum
    :param septum_depth: Strength of concavity in each chamber wall
    :return: Chamber mesh arrays Xc, Yc, Zc, Cc and triangle index arrays Ic, Jc, Kc
    """

    # Unpack the outer shell mesh.
    # X, Y, Z contain all the surface vertices for all aperture rings.
    # The remaining values are ignored here because chamber construction
    # only needs the shell geometry, not its colour or triangle faces.
    X, Y, Z, _, _, _, _ = shell_mesh

    # Number of growth stages / aperture rings in the shell.
    n_spiral = len(theta)

    # Chamber mesh vertices.
    Xc = []
    Yc = []
    Zc = []

    # Chamber colour/intensity values.
    Cc = []

    # Chamber triangle index arrays for Plotly Mesh3d.
    Ic = []
    Jc = []
    Kc = []

    # Track the theta value where the previous chamber was inserted.
    # New chamber walls are added once growth has advanced by chamber_spacing.
    last_chamber_theta = theta[0]

    # Walk along the shell growth path.
    # Each index s corresponds to one aperture ring in the outer shell mesh.
    for s in range(1, n_spiral - 1):

        # Add a chamber wall only at selected intervals.
        if theta[s] - last_chamber_theta >= chamber_spacing:

            # Locate the vertices belonging to the aperture ring at this
            # growth stage. This ring becomes the outer edge of the septum.
            start_index = s * aperture_points

            ring_x = X[start_index:start_index + aperture_points]
            ring_y = Y[start_index:start_index + aperture_points]
            ring_z = Z[start_index:start_index + aperture_points]

            # Estimate the centre of the aperture ring.
            # The septum will be built inward from this outer ring.
            centre_x = np.mean(ring_x)
            centre_y = np.mean(ring_y)
            centre_z = np.mean(ring_z)

            # -------------------------------------------------------------
            # Estimate local growth direction
            # -------------------------------------------------------------
            # To curve the septum, we need to know which way is "backward"
            # into the older part of the shell.
            #
            # We estimate the growth direction by comparing the mean position
            # of nearby aperture rings before and after the current one.
            if s > 0:
                dx = (
                    np.mean(X[(s+1)*aperture_points:(s+2)*aperture_points])
                    - np.mean(X[(s-1)*aperture_points:s*aperture_points])
                )
                dy = (
                    np.mean(Y[(s+1)*aperture_points:(s+2)*aperture_points])
                    - np.mean(Y[(s-1)*aperture_points:s*aperture_points])
                )
                dz = (
                    np.mean(Z[(s+1)*aperture_points:(s+2)*aperture_points])
                    - np.mean(Z[(s-1)*aperture_points:s*aperture_points])
                )
            else:
                dx, dy, dz = 0, 0, 1

            direction = np.array([dx, dy, dz])

            # Normalise the direction vector so it only represents direction,
            # not the distance between aperture rings.
            direction = direction / np.linalg.norm(direction)

            # Approximate aperture size.
            # This gives a scale for how deeply the septum should curve.
            aperture_radius = np.mean(
                np.sqrt(
                    (ring_x - centre_x)**2 +
                    (ring_y - centre_y)**2 +
                    (ring_z - centre_z)**2
                )
            )

            # Starting index of this septum's vertices in the chamber mesh.
            base_index = len(Xc)

            # -------------------------------------------------------------
            # Build one curved septum from nested rings
            # -------------------------------------------------------------
            # t = 0 gives the centre of the septum.
            # t = 1 gives the original aperture ring.
            #
            # By creating intermediate rings, we can make a smooth curved
            # surface instead of a single flat fan of triangles.
            for ring_idx in range(septum_rings + 1):

                t = ring_idx / septum_rings

                # Bowl curve:
                # - deepest at the centre
                # - smoothly approaches zero at the aperture edge
                #
                # This pushes the septum inward toward older shell growth.
                curve_offset = -septum_depth * aperture_radius * (1 - t**2)

                for p in range(aperture_points):

                    # Interpolate between the centre and the aperture rim.
                    # This creates one nested ring within the septum.
                    px = centre_x + t * (ring_x[p] - centre_x)
                    py = centre_y + t * (ring_y[p] - centre_y)
                    pz = centre_z + t * (ring_z[p] - centre_z)

                    # Push the point backward along the growth direction,
                    # producing the shallow concavity of the chamber wall.
                    px += curve_offset * direction[0]
                    py += curve_offset * direction[1]
                    pz += curve_offset * direction[2]

                    Xc.append(px)
                    Yc.append(py)
                    Zc.append(pz)

                    # Colour by growth age.
                    # Inner/older chambers have smaller growth_fraction.
                    # Outer/newer chambers have larger growth_fraction.
                    growth_fraction = s / (n_spiral - 1)

                    # Older inner chambers darker, newer outer chambers lighter.
                    chamber_colour = 0.4 + 0.6 * growth_fraction

                    Cc.append(chamber_colour)

            # -------------------------------------------------------------
            # Stitch the nested septum rings into a curved surface
            # -------------------------------------------------------------
            # Each pair of neighbouring nested rings forms a band of the
            # chamber wall. Each band is split into triangles.
            for ring_idx in range(septum_rings):

                ring_a = base_index + ring_idx * aperture_points
                ring_b = base_index + (ring_idx + 1) * aperture_points

                for p in range(aperture_points):

                    a0 = ring_a + p
                    a1 = ring_a + ((p + 1) % aperture_points)
                    b0 = ring_b + p
                    b1 = ring_b + ((p + 1) % aperture_points)

                    # First triangle in this small septum patch.
                    Ic.append(a0)
                    Jc.append(b0)
                    Kc.append(a1)

                    # Second triangle in this small septum patch.
                    Ic.append(a1)
                    Jc.append(b0)
                    Kc.append(b1)

            # Record where this chamber was added so the next one is spaced
            # further along the shell's growth path.
            last_chamber_theta = theta[s]

    return np.array(Xc), np.array(Yc), np.array(Zc), np.array(Cc), Ic, Jc, Kc

In [ ]:
import plotly.graph_objects as go


def plot_shell_mesh(
    shell_mesh,
    shell_opacity=1.0,
    septa_opacity=0.75,
    chamber_mesh=None,
    title="3D Logarithmic Shell Mesh"
):
    """
    Render the procedural shell geometry using Plotly Mesh3d.

    Conceptually, this function visualises two interacting structures:

    1. The outer shell surface
       - built from a sequence of connected aperture rings
       - representing the accumulated shell deposited during growth

    2. Optional internal chamber septa
       - curved internal walls inserted periodically during growth
       - representing the compartmentalised interior of a cephalopod shell

    The shell surface and internal chambers are rendered as separate meshes,
    allowing transparency and lighting to reveal the shell's internal
    architecture.

    Colour values are interpreted as shell pigmentation patterns generated
    during growth.

    Transparency can be used to create:
    - opaque shell reconstructions
    - translucent museum-style cutaways
    - exposed chamber visualisations

    :param shell_mesh: Tuple returned by build_shell_mesh
    :param shell_opacity: Opacity of the outer shell surface
    :param septa_opacity: Opacity of internal chamber septa
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param title: Plot title
    """

    # ---------------------------------------------------------------------
    # Unpack the outer shell mesh
    # ---------------------------------------------------------------------
    #
    # X, Y, Z:
    #   vertex positions for the shell surface
    #
    # C:
    #   per-vertex colour/intensity values used for shell pigmentation
    #
    # I, J, K:
    #   triangle index arrays defining the shell surface geometry
    X, Y, Z, C, I, J, K = shell_mesh

    # ---------------------------------------------------------------------
    # Create the outer shell surface
    # ---------------------------------------------------------------------
    #
    # Plotly Mesh3d renders the shell as many connected triangles.
    #
    # The shell colour comes from the generated pigmentation bands.
    #
    # Lighting parameters are tuned to give a nacreous / shell-like sheen
    # rather than a matte engineering-style surface.
    traces = [
        go.Mesh3d(
            x=X,
            y=Y,
            z=Z,

            # Triangle vertex indices
            i=I,
            j=J,
            k=K,

            # Procedural shell pigmentation
            intensity=C,
            colorscale="YlOrBr",
            showscale=False,

            # Transparency allows internal chambers to be revealed
            opacity=shell_opacity,

            # Smooth shading produces a more organic shell surface
            flatshading=False,

            # Lighting tuned for shell-like highlights and soft reflections
            lighting=dict(
                ambient=0.35,
                diffuse=0.95,
                specular=1.2,
                roughness=0.12,
                fresnel=0.5
            )
        )
    ]

    # ---------------------------------------------------------------------
    # Add internal chamber septa if present
    # ---------------------------------------------------------------------
    #
    # The chamber mesh is rendered separately from the shell surface so
    # transparency and colouring can differ between shell wall and chambers.
    if chamber_mesh is not None:

        # Chamber mesh vertices, colours and triangle indices
        Xc, Yc, Zc, Cc, Ic, Jc, Kc = chamber_mesh

        traces.append(
            go.Mesh3d(
                x=Xc,
                y=Yc,
                z=Zc,

                i=Ic,
                j=Jc,
                k=Kc,

                # Chamber colour/intensity values.
                # Older chambers can be darker than newer ones.
                intensity=Cc,

                colorscale="YlOrBr",

                # Chamber transparency helps preserve visibility through
                # translucent outer shell walls.
                opacity=septa_opacity,

                flatshading=False,

                showscale=False
            )
        )

    # ---------------------------------------------------------------------
    # Assemble the full figure
    # ---------------------------------------------------------------------
    fig = go.Figure(data=traces)

    # ---------------------------------------------------------------------
    # Configure the viewing scene
    # ---------------------------------------------------------------------
    #
    # aspectmode="data" preserves the shell proportions correctly.
    #
    # Without this, Plotly may distort the shell geometry to fit the figure.
    fig.update_layout(
        title=title,

        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",

            # Preserve true shell proportions
            aspectmode="data"
        ),

        width=900,
        height=800
    )

    # Display the interactive shell visualisation.
    #
    # The resulting figure can be:
    # - rotated
    # - zoomed
    # - inspected interactively
    #
    # revealing both external morphology and internal chamber structure.
    fig.show()

In [ ]:
theta, r, x, y, width, height = logarithmic_spiral(
    a=1.0,
    b=0.13,
    turns=4,
    points=500,
    aperture_mode="abutting",
    abutment_scale=1.55,
    abutment_height_ratio=0.7
)

shell_mesh = build_shell_mesh(
    theta,
    r,
    x,
    y,
    width,
    height,
    z_scale=0.0,
    aperture_points=48,
    ribbing_amplitude=0.06,
    ribbing_frequency=18,
    lip_start=0.88,
    lip_strength=0.35,
    lip_power=3
)

chamber_mesh = build_chamber_septa(
    theta,
    shell_mesh,
    aperture_points=48,
    chamber_spacing=0.65,
    septum_rings=10,
    septum_depth=0.35
)

plot_shell_mesh(shell_mesh, 1.0, 0.55, chamber_mesh)
plot_shell_mesh(shell_mesh, 0.25, 0.75, chamber_mesh)